# Aula03 — LangChain

Uma seção por vídeo, cada uma autossuficiente. Requer o arquivo `.env` (veja `.example.env`).
Os `main.py` originais continuam em `video0X/`.

## video01

In [ ]:
import os
import re

from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# Tenta usar embeddings do Google (Gemini); usa OpenAI como fallback se não disponível
try:
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    _HAS_GOOGLE_EMB = True
except Exception:
    _HAS_GOOGLE_EMB = False

try:
    from langchain_openai import OpenAIEmbeddings
    _HAS_OPENAI_EMB = True
except Exception:
    _HAS_OPENAI_EMB = False

MODELO_GEMINI_PADRAO = "gemini-embedding-001"

def _mascarar_segredos(texto: str) -> str:
    """Remove chaves de API que possam aparecer em mensagens de erro antes de imprimir."""
    return re.sub(r"(key=|AIza)[A-Za-z0-9_\-]{6,}", r"\1<REDACTED>", str(texto))

def _nomes_candidatos(modelo: str):
    """Gera as variações aceitas do nome do modelo.
 
    langchain-google-genai >= 3.x aceita o nome puro ('gemini-embedding-001');
    versões 2.x mandam a string crua para a API, que exige o prefixo 'models/'
    e devolve 'unexpected model name format' sem ele. Tentar as duas cobre
    ambos os casos sem depender da versão instalada.
    """
    if modelo.startswith("models/"):
        return [modelo, modelo.removeprefix("models/")]
    return [modelo, f"models/{modelo}"]

def _criar_embeddings_google():
    """Instancia e valida os embeddings do Gemini. Retorna None se não der para usar."""
    if not _HAS_GOOGLE_EMB:
        print(" Pacote langchain-google-genai não instalado.")
        return None
    if not os.environ.get("GOOGLE_API_KEY"):
        print(" GOOGLE_API_KEY não encontrada no ambiente (.env).")
        return None
 
    modelo = os.environ.get("GOOGLE_EMBEDDING_MODEL", MODELO_GEMINI_PADRAO)
 
    for nome in _nomes_candidatos(modelo):
        try:
            embeddings = GoogleGenerativeAIEmbeddings(model=nome)
            # Chamada mínima só para validar credencial + nome do modelo antes de indexar.
            embeddings.embed_query("ping")
            print(f" Usando GoogleGenerativeAIEmbeddings (modelo: {nome})")
            return embeddings
        except Exception as e:
            print(f" Modelo '{nome}' indisponível -> {type(e).__name__}: {_mascarar_segredos(e)}")
 
    return None

def _criar_embeddings_openai():
    """Fallback: embeddings da OpenAI. Retorna None se não der para usar."""
    if not _HAS_OPENAI_EMB or not os.environ.get("OPENAI_API_KEY"):
        return None
    try:
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        print(" Usando OpenAIEmbeddings como fallback")
        return embeddings
    except Exception as e:
        print(" Falha ao inicializar OpenAIEmbeddings:", _mascarar_segredos(e))
        return None

def criar_embeddings():
    """Escolhe o provedor de embeddings: Gemini primeiro, OpenAI como reserva."""
    return _criar_embeddings_google() or _criar_embeddings_openai()

def run_aula_1():
    print("--- INICIANDO AULA 3: PIPELINE DE INGESTÃO LOCAL ---")

    # 1. Dados corporativos simulados (Relatório Financeiro)
    documento_corporativo = """
    Relatório de Desempenho Financeiro - Indústrias ACME (Maio 2026).
    O faturamento consolidado no último mês atingiu R$ 12.5 milhões, representando uma alta de 8% em relação a abril.
    O principal motor de crescimento foi a divisão de Microcontroladores, que expandiu sua operação global em 15%.
    As despesas operacionais (OPEX) totalizaram R$ 4.2 milhões, mantendo-se dentro do orçamento previsto.
    A projeção para o próximo trimestre é manter a taxa de crescimento acima de dois dígitos.
    """

    # 2. Chunking (Fragmentação Inteligente) com Overlap para não perder contexto
    print("\n[Passo 1] Fragmentando o texto...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=150,
        chunk_overlap=30
    )
    chunks = text_splitter.split_text(documento_corporativo)
    for i, chunk in enumerate(chunks):
        print(f" Chunk {i+1}: {chunk.strip()}")

    # 3. Inicialização do modelo de Embedding
    print("\n[Passo 2] Gerando vetores (Gemini/Google ou OpenAI, conforme disponível)...")
    embeddings = criar_embeddings()
    if embeddings is None:
        print("ERRO: Nenhum provedor de embeddings disponível. "
              "Configure GOOGLE_API_KEY (ou OPENAI_API_KEY) e as bibliotecas necessárias.")
        return

    # 4. Criação do Banco Vetorial local com FAISS
    print("\n[Passo 3] Indexando vetores no FAISS...")
    try:
        vector_store = FAISS.from_texts(chunks, embeddings)
    except Exception as e:
        print("ERRO ao indexar os vetores:", _mascarar_segredos(e))
        return

    # 5. Teste de Busca Semântica (Simulando a pergunta do usuário)
    query = "Quanto a empresa faturou no mês passado?"
    print(f"\n[Passo 4] Executando busca semântica para: '{query}'")

    resultados = vector_store.similarity_search(query, k=1)

    print("\n[Resultado da Busca] Documento mais relevante encontrado:")
    print(f" -> {resultados[0].page_content.strip()}")

### Execução

In [ ]:
load_dotenv()
run_aula_1()

## video02

In [ ]:
import os
import re

from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

MODELO_EMBEDDING_PADRAO = "gemini-embedding-001"
MODELO_CHAT_PADRAO = "gemini-3.5-flash"
MODELO_OPENAI_PADRAO = os.environ.get("OPENAI_CHAT_MODEL", "")

QTD_VARIACOES = 2
DOCS_POR_CONSULTA = 2

def _mascarar_segredos(texto: str) -> str:
    """Remove chaves de API que possam aparecer em mensagens de erro antes de imprimir."""
    return re.sub(r"(key=|AIza)[A-Za-z0-9_\-]{6,}", r"\1<REDACTED>", str(texto))


def _nomes_candidatos(modelo: str):
    """Nome puro e com prefixo 'models/'.

    langchain-google-genai >= 3.x normaliza sozinho; a 2.x manda a string crua
    para a API, que exige o prefixo e devolve 'unexpected model name format'.
    """
    if modelo.startswith("models/"):
        return [modelo, modelo.removeprefix("models/")]
    return [modelo, f"models/{modelo}"]

def criar_embeddings():
    """Instancia e valida os embeddings do Gemini antes de indexar qualquer coisa."""
    modelo = os.environ.get("GOOGLE_EMBEDDING_MODEL", MODELO_EMBEDDING_PADRAO)

    for nome in _nomes_candidatos(modelo):
        try:
            embeddings = GoogleGenerativeAIEmbeddings(model=nome)
            embeddings.embed_query("ping")  # valida credencial + nome do modelo
            print(f" Embeddings OK (modelo: {nome})")
            return embeddings
        except Exception as e:
            print(f" Modelo '{nome}' indisponível -> {type(e).__name__}: {_mascarar_segredos(e)}")

    return None

def _extrair_variacoes(texto: str, limite: int):
    """Transforma a resposta da LLM em uma lista de perguntas, sem numeração nem linhas vazias."""
    variacoes = []
    for linha in texto.splitlines():
        limpa = re.sub(r"^\s*(\d+[\.\)]|[-*•])\s*", "", linha).strip()
        if limpa:
            variacoes.append(limpa)
    return variacoes[:limite]

def buscar_multi_query(vector_store, consultas, k: int):
    """Busca cada consulta e devolve a união dos documentos, sem repetidos.

    O dict preserva a ordem de inserção, então os documentos da pergunta
    original aparecem primeiro no contexto.
    """
    unicos = {}
    for consulta in consultas:
        for doc in vector_store.similarity_search(consulta, k=k):
            unicos.setdefault(doc.page_content, doc)
    return list(unicos.values())

def run_aula_2():
    print("--- INICIANDO AULA 2: ADVANCED RAG (MULTI-QUERY) ---")

    embeddings = criar_embeddings()
    if embeddings is None:
        print("ERRO: não foi possível inicializar os embeddings. Verifique GOOGLE_API_KEY no .env.")
        return

    llm = ChatGoogleGenerativeAI(model=os.environ.get("GOOGLE_CHAT_MODEL", MODELO_CHAT_PADRAO))

    # 1. Base de conhecimento local pré-indexada
    textos = [
        "O faturamento consolidado da ACME em Maio de 2026 foi de R$ 12.5 milhões.",
        "A divisão de Microcontroladores teve expansão de 15% na operação global.",
        "O Diretor Executivo da ACME em 2026 é o Engenheiro Carlos Eduardo."
    ]

    try:
        vector_store = FAISS.from_texts(textos, embeddings)
    except Exception as e:
        print("ERRO ao indexar os vetores:", _mascarar_segredos(e))
        return

    # 2. Query Transformation: a LLM gera variações para aumentar o Recall
    pergunta_usuario = "Quem lidera a empresa ACME e como foi o lucro?"
    print(f"\n[Pergunta Original]: {pergunta_usuario}")

    prompt_multi_query = PromptTemplate.from_template(
        "Você é um assistente técnico de IA. Gere {qtd} variações alternativas da seguinte pergunta "
        "para ajudar na busca em um banco de dados vetorial.\n"
        "Responda APENAS com as perguntas, uma por linha, sem numeração e sem comentários.\n"
        "Pergunta original: {pergunta}"
    )

    chain_variacoes = prompt_multi_query | llm | StrOutputParser()

    try:
        resposta_variacoes = chain_variacoes.invoke(
            {"pergunta": pergunta_usuario, "qtd": QTD_VARIACOES}
        )
    except Exception as e:
        print("ERRO ao gerar variações:", _mascarar_segredos(e))
        return

    variacoes = _extrair_variacoes(resposta_variacoes, QTD_VARIACOES)

    print("\n[Query Transformation - Variações Geradas pela LLM]:")
    for v in variacoes:
        print(f" - {v}")

    # 3. Retrieval: busca a pergunta original E cada variação, unindo os resultados
    print("\n[Retrieval] Buscando informações relevantes...")
    consultas = [pergunta_usuario] + variacoes
    documentos_recuperados = buscar_multi_query(vector_store, consultas, k=DOCS_POR_CONSULTA)

    contexto = "\n".join(doc.page_content for doc in documentos_recuperados)
    print(f"{len(documentos_recuperados)} documento(s) único(s) recuperado(s):\n{contexto}")

    # 4. Generation: resposta fundamentada apenas no contexto recuperado
    prompt_rag = PromptTemplate.from_template(
        "Você é um assistente corporativo. Responda à pergunta do usuário utilizando APENAS o contexto fornecido abaixo.\n"
        "Se não souber a resposta com base no contexto, diga honestamente que não encontrou a informação.\n\n"
        "Contexto:\n{contexto}\n\n"
        "Pergunta: {pergunta}\n\n"
        "Resposta Resumida e Direta:"
    )

    chain_rag = prompt_rag | llm | StrOutputParser()

    try:
        resposta_final = chain_rag.invoke({"contexto": contexto, "pergunta": pergunta_usuario})
    except Exception as e:
        print("ERRO ao gerar a resposta final:", _mascarar_segredos(e))
        return

    print("\n[Resposta Final Fundamentada da LLM]:")
    print(resposta_final)

### Execução

In [ ]:
load_dotenv()
run_aula_2()

## video03

In [ ]:
import os
from pathlib import Path
import re
import sqlite3
from contextlib import closing

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.chat_history import BaseChatMessageHistory

# Fallback opcional: só é usado se o pacote estiver instalado.
try:
    from langchain_openai import ChatOpenAI
    _HAS_OPENAI = True
except Exception:
    _HAS_OPENAI = False

MODELO_CHAT_PADRAO = os.environ.get("GEMINI_CHAT_MODEL", "")
MODELO_OPENAI_PADRAO = os.environ.get("OPENAI_CHAT_MODEL", "")


# Arquivo único no diretório do projeto. Dá para abrir no VS Code e ver as linhas.
DATABASE = os.environ.get("SQLITE_PATH", "memoria_chat.db")
CAMINHO_BANCO = Path(Path(__file__).parent, DATABASE)
CAMINHO_BANCO.touch(exist_ok=True)

TABELA_HISTORICO = "historico_conversas"

TIPOS_PERSISTIDOS = {"human": HumanMessage, "ai": AIMessage}


def _mascarar_segredos(texto: str) -> str:
    """Remove credenciais que possam aparecer em mensagens de erro antes de imprimir."""
    return re.sub(r"(key=|AIza|sk-)[A-Za-z0-9_\-]{6,}", r"\1<REDACTED>", str(texto))


def _validar(llm, rotulo: str):
    """Faz uma chamada mínima para confirmar que o provedor responde.

    É aqui que a cota estourada aparece (429), em vez de estourar no meio da
    conversa depois de já ter gravado o turno do usuário no banco.
    """
    try:
        llm.invoke("ping")
        print(f" LLM OK: {rotulo}")
        return llm
    except Exception as e:
        print(f" {rotulo} indisponível -> {type(e).__name__}: {_mascarar_segredos(e)}")
        return None

def _criar_llm_google():
    if not os.environ.get("GOOGLE_API_KEY"):
        print(" GOOGLE_API_KEY não encontrada no .env.")
        return None
    modelo = os.environ.get("GOOGLE_CHAT_MODEL", MODELO_CHAT_PADRAO)
    return _validar(ChatGoogleGenerativeAI(model=modelo), f"Google / {modelo}")


def _criar_llm_openai():
    if not _HAS_OPENAI:
        print(" Pacote langchain-openai não instalado.")
        return None
    if not os.environ.get("OPENAI_API_KEY"):
        print(" OPENAI_API_KEY não encontrada no .env.")
        return None
    modelo = os.environ.get("OPENAI_CHAT_MODEL", MODELO_OPENAI_PADRAO)
    return _validar(ChatOpenAI(model=modelo), f"OpenAI / {modelo}")


def criar_llm():
    """Escolhe o provedor do chat.

    LLM_PROVIDER aceita 'google', 'openai' ou 'auto' (padrão): tenta o Google e
    cai para a OpenAI se ele não responder — cota estourada, chave ausente ou
    modelo indisponível.
    """
    provedor = os.environ.get("LLM_PROVIDER", "auto").lower()

    if provedor == "google":
        return _criar_llm_google()
    if provedor == "openai":
        return _criar_llm_openai()

    return _criar_llm_google() or _criar_llm_openai()

class SQLiteChatMessageHistory(BaseChatMessageHistory):
    """Persistência de mensagens em SQLite.

    Abre uma conexão por operação e fecha em seguida. Uma conexão sqlite3 não
    pode ser usada em uma thread diferente da que a criou, e o LangChain executa
    a chain em uma thread pool — por isso nada de conexão guardada em atributo.
    Abrir conexão no SQLite é só abrir um arquivo local, então o custo é
    irrelevante (diferente do MySQL, onde a conexão é um socket de rede).
    """

    def __init__(self, session_id: str, caminho: str = None):
        self.session_id = session_id
        self.caminho = caminho or CAMINHO_BANCO
        self._garantir_tabela()

    def _conectar(self) -> sqlite3.Connection:
        conexao = sqlite3.connect(self.caminho, timeout=5.0)
        conexao.execute("PRAGMA journal_mode=WAL")  # evita 'database is locked'
        return conexao

    def _garantir_tabela(self) -> None:
        with closing(self._conectar()) as conexao, conexao:
            conexao.execute(
                f"""
                CREATE TABLE IF NOT EXISTS {TABELA_HISTORICO} (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    role TEXT NOT NULL CHECK (role IN ('human', 'ai')),
                    content TEXT NOT NULL,
                    criado_em TEXT DEFAULT CURRENT_TIMESTAMP
                )
                """
            )
            conexao.execute(
                f"CREATE INDEX IF NOT EXISTS idx_sessao ON {TABELA_HISTORICO} (session_id, id)"
            )

    @property
    def messages(self) -> list[BaseMessage]:
        with closing(self._conectar()) as conexao:
            linhas = conexao.execute(
                f"SELECT role, content FROM {TABELA_HISTORICO} "
                "WHERE session_id = ? ORDER BY id ASC",
                (self.session_id,),
            ).fetchall()

        return [
            TIPOS_PERSISTIDOS[role](content=content)
            for role, content in linhas
            if role in TIPOS_PERSISTIDOS
        ] 

    
    def add_messages(self, messages: list[BaseMessage]) -> None:
        linhas = [
            (self.session_id, m.type, m.content)
            for m in messages
            if m.type in TIPOS_PERSISTIDOS
        ]
        if not linhas:
            return

        # 'with conexao' abre transação: commit no sucesso, rollback na exceção.
        with closing(self._conectar()) as conexao, conexao:
            conexao.executemany(
                f"INSERT INTO {TABELA_HISTORICO} (session_id, role, content) VALUES (?, ?, ?)",
                linhas,
            )

    
    def clear(self) -> None:
        with closing(self._conectar()) as conexao, conexao:
            conexao.execute(
                f"DELETE FROM {TABELA_HISTORICO} WHERE session_id = ?",
                (self.session_id,),
            )

def run_aula_3_sqlite():
    print("--- INICIANDO AULA 3: MEMÓRIA PERSISTENTE COM SQLITE ---")
    print(f"Banco: {CAMINHO_BANCO}")

    llm = criar_llm()
    if llm is None:
        print("ERRO: nenhum provedor de LLM disponível. Verifique as chaves no .env.")
        return

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Você é um assistente técnico de banco de dados. Seja breve e amigável."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}")
    ])

    chain = prompt | llm

    chain_com_memoria = RunnableWithMessageHistory(
        chain,
        get_session_history=SQLiteChatMessageHistory,
        input_messages_key="input",
        history_messages_key="history"
    )

    configuracao_usuario = {"configurable": {"session_id": "sessao_aluno_pos_tech"}}

    # ---- TURNO 1 ----
    print("\n[Turno 1] Enviando informações iniciais...")
    input_1 = "Olá professor! Meu nome é Lucas e estou desenvolvendo o projeto final em Python."
    print(f"Aluno: {input_1}")
    resposta_1 = chain_com_memoria.invoke({"input": input_1}, config=configuracao_usuario)
    print(f"Assistente: {resposta_1.content}")

    # ---- TURNO 2 (teste de retenção via banco) ----
    print("\n[Turno 2] Perguntando com base no histórico gravado no SQLite...")
    input_2 = "Com base no que eu te disse, qual é o meu nome e qual linguagem estou usando?"
    print(f"Aluno: {input_2}")
    resposta_2 = chain_com_memoria.invoke({"input": input_2}, config=configuracao_usuario)
    print(f"Assistente: {resposta_2.content}")

    print(f"\n--- Concluído. Abra o arquivo {CAMINHO_BANCO} no VS Code para ver as linhas gravadas. ---")

### Execução

In [ ]:
load_dotenv()

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("OPENAI_API_KEY")):
    print("ERRO: configure GOOGLE_API_KEY ou OPENAI_API_KEY no .env antes de rodar o script.")
else:
    try:
        run_aula_3_sqlite()
    except Exception as e:
        print(f"\nERRO na execução: {type(e).__name__}: {_mascarar_segredos(e)}")